# Tackling Mode Collapse in GANs
## DCGAN vs WGAN-GP Comparison
**22F-3396 & 22F-3369 | GenAI Assignment 03**

---

## 1. Environment Setup & Imports

In [ ]:
import os, time, json, math
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
import torchvision
import torchvision.transforms as transforms
import torchvision.utils as vutils
from torchvision.datasets import ImageFolder

# Config
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. Hyperparameters

In [ ]:
# Hyperparameters
IMAGE_SIZE = 64
BATCH_SIZE = 64
NZ = 100          # latent vector size
NGF = 64          # generator feature maps
NDF = 64          # discriminator feature maps
NC = 3            # color channels
NUM_EPOCHS_DCGAN = 50
NUM_EPOCHS_WGAN = 50
LR = 0.0002
BETA1 = 0.5
BETA2 = 0.999
LAMBDA_GP = 10    # gradient penalty coefficient
N_CRITIC = 5      # critic updates per generator update
SAVE_EVERY = 5    # checkpoint interval

## 3. Data Preparation

In [ ]:
# === DATASET PATHS (adjust for your Kaggle environment) ===
# For Anime Faces:
DATASET_PATH = '/kaggle/input/anime-faces/data'
# For Pokemon Sprites (uncomment if using):
# DATASET_PATH = '/kaggle/input/pokemon-sprites/pokemon'

transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),  # normalize to [-1, 1]
])

dataset = ImageFolder(root=DATASET_PATH, transform=transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=2, pin_memory=True, drop_last=True)

print(f'Dataset size: {len(dataset)} images')
print(f'Batches per epoch: {len(dataloader)}')

# Show sample real images
real_batch = next(iter(dataloader))
plt.figure(figsize=(10, 10))
plt.axis('off')
plt.title('Real Training Images')
plt.imshow(np.transpose(vutils.make_grid(real_batch[0][:64], padding=2, normalize=True).cpu(), (1,2,0)))
plt.show()

## 4. Weight Initialization

In [ ]:
def weights_init(m):
    """Custom weight initialization for Conv and BatchNorm layers."""
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

## 5. DCGAN Architecture

In [ ]:
class DCGANGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            # input: z (NZ x 1 x 1)
            nn.ConvTranspose2d(NZ, NGF*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(NGF*8),
            nn.ReLU(True),
            # state: (NGF*8) x 4 x 4
            nn.ConvTranspose2d(NGF*8, NGF*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF*4),
            nn.ReLU(True),
            # state: (NGF*4) x 8 x 8
            nn.ConvTranspose2d(NGF*4, NGF*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF*2),
            nn.ReLU(True),
            # state: (NGF*2) x 16 x 16
            nn.ConvTranspose2d(NGF*2, NGF, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF),
            nn.ReLU(True),
            # state: (NGF) x 32 x 32
            nn.ConvTranspose2d(NGF, NC, 4, 2, 1, bias=False),
            nn.Tanh()
            # output: (NC) x 64 x 64
        )

    def forward(self, x):
        return self.main(x)


class DCGANDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            # input: (NC) x 64 x 64
            nn.Conv2d(NC, NDF, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # state: (NDF) x 32 x 32
            nn.Conv2d(NDF, NDF*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NDF*2),
            nn.LeakyReLU(0.2, inplace=True),
            # state: (NDF*2) x 16 x 16
            nn.Conv2d(NDF*2, NDF*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NDF*4),
            nn.LeakyReLU(0.2, inplace=True),
            # state: (NDF*4) x 8 x 8
            nn.Conv2d(NDF*4, NDF*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NDF*8),
            nn.LeakyReLU(0.2, inplace=True),
            # state: (NDF*8) x 4 x 4
            nn.Conv2d(NDF*8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.main(x).view(-1, 1).squeeze(1)


print('DCGAN Generator:')
netG_dc = DCGANGenerator().to(device)
netG_dc.apply(weights_init)
print(f'  Parameters: {sum(p.numel() for p in netG_dc.parameters()):,}')

print('DCGAN Discriminator:')
netD_dc = DCGANDiscriminator().to(device)
netD_dc.apply(weights_init)
print(f'  Parameters: {sum(p.numel() for p in netD_dc.parameters()):,}')

## 6. WGAN-GP Architecture

In [ ]:
class WGANGenerator(nn.Module):
    """Same architecture as DCGAN Generator."""
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(NZ, NGF*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(NGF*8),
            nn.ReLU(True),
            nn.ConvTranspose2d(NGF*8, NGF*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF*4),
            nn.ReLU(True),
            nn.ConvTranspose2d(NGF*4, NGF*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF*2),
            nn.ReLU(True),
            nn.ConvTranspose2d(NGF*2, NGF, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF),
            nn.ReLU(True),
            nn.ConvTranspose2d(NGF, NC, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.main(x)


class WGANCritic(nn.Module):
    """Critic (no Sigmoid, no BatchNorm — uses InstanceNorm instead)."""
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(NC, NDF, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(NDF, NDF*2, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(NDF*2, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(NDF*2, NDF*4, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(NDF*4, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(NDF*4, NDF*8, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(NDF*8, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(NDF*8, 1, 4, 1, 0, bias=False),
            # NO Sigmoid — raw critic score
        )

    def forward(self, x):
        return self.main(x).view(-1, 1).squeeze(1)


print('WGAN-GP Generator:')
netG_wg = WGANGenerator().to(device)
netG_wg.apply(weights_init)
print(f'  Parameters: {sum(p.numel() for p in netG_wg.parameters()):,}')

print('WGAN-GP Critic:')
netC_wg = WGANCritic().to(device)
netC_wg.apply(weights_init)
print(f'  Parameters: {sum(p.numel() for p in netC_wg.parameters()):,}')

## 7. Gradient Penalty (for WGAN-GP)

In [ ]:
def compute_gradient_penalty(critic, real_samples, fake_samples, device):
    """Compute gradient penalty for WGAN-GP."""
    batch_size = real_samples.size(0)
    alpha = torch.rand(batch_size, 1, 1, 1, device=device)
    interpolates = (alpha * real_samples + (1 - alpha) * fake_samples).requires_grad_(True)
    
    d_interpolates = critic(interpolates)
    
    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=torch.ones_like(d_interpolates),
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]
    
    gradients = gradients.view(batch_size, -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty

## 8. Training — DCGAN

In [ ]:
# DCGAN Training
criterion = nn.BCELoss()
optimizerD_dc = optim.Adam(netD_dc.parameters(), lr=LR, betas=(BETA1, BETA2))
optimizerG_dc = optim.Adam(netG_dc.parameters(), lr=LR, betas=(BETA1, BETA2))
scaler_D = GradScaler()
scaler_G = GradScaler()

fixed_noise = torch.randn(64, NZ, 1, 1, device=device)

G_losses_dc, D_losses_dc = [], []
img_list_dc = []

print('Starting DCGAN Training...')
start_time = time.time()

for epoch in range(NUM_EPOCHS_DCGAN):
    epoch_d_loss, epoch_g_loss = 0.0, 0.0
    for i, (real_imgs, _) in enumerate(dataloader):
        real_imgs = real_imgs.to(device)
        b_size = real_imgs.size(0)
        real_label = torch.ones(b_size, device=device)
        fake_label = torch.zeros(b_size, device=device)

        # --- Train Discriminator ---
        netD_dc.zero_grad()
        with autocast():
            output_real = netD_dc(real_imgs)
            errD_real = criterion(output_real, real_label)
            noise = torch.randn(b_size, NZ, 1, 1, device=device)
            fake_imgs = netG_dc(noise)
            output_fake = netD_dc(fake_imgs.detach())
            errD_fake = criterion(output_fake, fake_label)
            errD = errD_real + errD_fake
        scaler_D.scale(errD).backward()
        scaler_D.step(optimizerD_dc)
        scaler_D.update()

        # --- Train Generator ---
        netG_dc.zero_grad()
        with autocast():
            output = netD_dc(fake_imgs)
            errG = criterion(output, real_label)
        scaler_G.scale(errG).backward()
        scaler_G.step(optimizerG_dc)
        scaler_G.update()

        epoch_d_loss += errD.item()
        epoch_g_loss += errG.item()

    avg_d = epoch_d_loss / len(dataloader)
    avg_g = epoch_g_loss / len(dataloader)
    G_losses_dc.append(avg_g)
    D_losses_dc.append(avg_d)

    elapsed = time.time() - start_time
    print(f'[DCGAN] Epoch [{epoch+1}/{NUM_EPOCHS_DCGAN}] '
          f'D_loss: {avg_d:.4f}  G_loss: {avg_g:.4f}  '
          f'Time: {elapsed:.0f}s')

    # Save sample images
    if (epoch+1) % SAVE_EVERY == 0 or epoch == 0:
        with torch.no_grad():
            fake = netG_dc(fixed_noise).detach().cpu()
        img_list_dc.append(vutils.make_grid(fake[:16], padding=2, normalize=True))

    # Checkpoint
    if (epoch+1) % SAVE_EVERY == 0:
        torch.save({
            'epoch': epoch+1,
            'netG': netG_dc.state_dict(),
            'netD': netD_dc.state_dict(),
            'optimG': optimizerG_dc.state_dict(),
            'optimD': optimizerD_dc.state_dict(),
            'G_losses': G_losses_dc,
            'D_losses': D_losses_dc,
        }, f'dcgan_checkpoint_epoch{epoch+1}.pth')

    # Memory cleanup
    torch.cuda.empty_cache()

print(f'DCGAN Training Complete! Total time: {time.time()-start_time:.0f}s')

## 9. Training — WGAN-GP

In [ ]:
# WGAN-GP Training
optimizerC_wg = optim.Adam(netC_wg.parameters(), lr=LR, betas=(BETA1, BETA2))
optimizerG_wg = optim.Adam(netG_wg.parameters(), lr=LR, betas=(BETA1, BETA2))

fixed_noise_wg = torch.randn(64, NZ, 1, 1, device=device)

G_losses_wg, C_losses_wg = [], []
img_list_wg = []

print('Starting WGAN-GP Training...')
start_time = time.time()

for epoch in range(NUM_EPOCHS_WGAN):
    epoch_c_loss, epoch_g_loss = 0.0, 0.0
    g_steps = 0
    
    for i, (real_imgs, _) in enumerate(dataloader):
        real_imgs = real_imgs.to(device)
        b_size = real_imgs.size(0)

        # --- Train Critic ---
        netC_wg.zero_grad()
        noise = torch.randn(b_size, NZ, 1, 1, device=device)
        fake_imgs = netG_wg(noise).detach()

        critic_real = netC_wg(real_imgs).mean()
        critic_fake = netC_wg(fake_imgs).mean()
        gp = compute_gradient_penalty(netC_wg, real_imgs.data, fake_imgs.data, device)

        # Wasserstein loss + gradient penalty
        errC = critic_fake - critic_real + LAMBDA_GP * gp
        errC.backward()
        optimizerC_wg.step()

        epoch_c_loss += errC.item()

        # --- Train Generator every N_CRITIC steps ---
        if (i + 1) % N_CRITIC == 0:
            netG_wg.zero_grad()
            noise = torch.randn(b_size, NZ, 1, 1, device=device)
            fake_imgs = netG_wg(noise)
            errG = -netC_wg(fake_imgs).mean()  # maximize critic score
            errG.backward()
            optimizerG_wg.step()
            epoch_g_loss += errG.item()
            g_steps += 1

    avg_c = epoch_c_loss / len(dataloader)
    avg_g = epoch_g_loss / max(g_steps, 1)
    C_losses_wg.append(avg_c)
    G_losses_wg.append(avg_g)

    elapsed = time.time() - start_time
    print(f'[WGAN-GP] Epoch [{epoch+1}/{NUM_EPOCHS_WGAN}] '
          f'C_loss: {avg_c:.4f}  G_loss: {avg_g:.4f}  '
          f'Time: {elapsed:.0f}s')

    if (epoch+1) % SAVE_EVERY == 0 or epoch == 0:
        with torch.no_grad():
            fake = netG_wg(fixed_noise_wg).detach().cpu()
        img_list_wg.append(vutils.make_grid(fake[:16], padding=2, normalize=True))

    if (epoch+1) % SAVE_EVERY == 0:
        torch.save({
            'epoch': epoch+1,
            'netG': netG_wg.state_dict(),
            'netC': netC_wg.state_dict(),
            'optimG': optimizerG_wg.state_dict(),
            'optimC': optimizerC_wg.state_dict(),
            'G_losses': G_losses_wg,
            'C_losses': C_losses_wg,
        }, f'wgangp_checkpoint_epoch{epoch+1}.pth')

    torch.cuda.empty_cache()

print(f'WGAN-GP Training Complete! Total time: {time.time()-start_time:.0f}s')

## 10. Training Loss Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# DCGAN losses
axes[0].set_title('DCGAN Training Losses', fontsize=14)
axes[0].plot(G_losses_dc, label='Generator', color='#2196F3', linewidth=2)
axes[0].plot(D_losses_dc, label='Discriminator', color='#FF5722', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# WGAN-GP losses
axes[1].set_title('WGAN-GP Training Losses', fontsize=14)
axes[1].plot(G_losses_wg, label='Generator', color='#4CAF50', linewidth=2)
axes[1].plot(C_losses_wg, label='Critic', color='#9C27B0', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_losses.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Generated Image Visualization

In [ ]:
def show_generated_images(generator, title, num_images=10, nrow=5):
    """Generate and display images from a trained generator."""
    generator.eval()
    with torch.no_grad():
        noise = torch.randn(num_images, NZ, 1, 1, device=device)
        fake_images = generator(noise).cpu()
    generator.train()
    
    grid = vutils.make_grid(fake_images, nrow=nrow, padding=2, normalize=True)
    plt.figure(figsize=(12, 6))
    plt.axis('off')
    plt.title(title, fontsize=16)
    plt.imshow(np.transpose(grid, (1, 2, 0)))
    plt.show()
    return fake_images

# Show DCGAN results
print('='*60)
print('DCGAN Generated Samples')
print('='*60)
dc_samples = show_generated_images(netG_dc, 'DCGAN — Generated Images', num_images=10)

# Show WGAN-GP results
print('='*60)
print('WGAN-GP Generated Samples')
print('='*60)
wg_samples = show_generated_images(netG_wg, 'WGAN-GP — Generated Images', num_images=10)

## 12. Side-by-Side Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Real images
real_grid = vutils.make_grid(real_batch[0][:16], nrow=4, padding=2, normalize=True)
axes[0].set_title('Real Images', fontsize=14, fontweight='bold')
axes[0].imshow(np.transpose(real_grid, (1,2,0)))
axes[0].axis('off')

# DCGAN
with torch.no_grad():
    noise = torch.randn(16, NZ, 1, 1, device=device)
    dc_grid = vutils.make_grid(netG_dc(noise).cpu(), nrow=4, padding=2, normalize=True)
axes[1].set_title('DCGAN Generated', fontsize=14, fontweight='bold')
axes[1].imshow(np.transpose(dc_grid, (1,2,0)))
axes[1].axis('off')

# WGAN-GP
with torch.no_grad():
    wg_grid = vutils.make_grid(netG_wg(noise).cpu(), nrow=4, padding=2, normalize=True)
axes[2].set_title('WGAN-GP Generated', fontsize=14, fontweight='bold')
axes[2].imshow(np.transpose(wg_grid, (1,2,0)))
axes[2].axis('off')

plt.suptitle('Real vs DCGAN vs WGAN-GP Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Training Progression (Epoch-wise)

In [ ]:
def show_progression(img_list, title):
    """Show how generated images improved over training."""
    if not img_list:
        print(f'No images saved for {title}')
        return
    cols = min(len(img_list), 5)
    fig, axes = plt.subplots(1, cols, figsize=(4*cols, 4))
    if cols == 1:
        axes = [axes]
    for idx, (ax, img) in enumerate(zip(axes, img_list[:cols])):
        ax.imshow(np.transpose(img, (1,2,0)))
        ax.axis('off')
        ax.set_title(f'Epoch {(idx)*SAVE_EVERY + 1 if idx > 0 else 1}')
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_progression(img_list_dc, 'DCGAN Training Progression')
show_progression(img_list_wg, 'WGAN-GP Training Progression')

## 14. Diversity Analysis

In [ ]:
def compute_diversity_score(generator, n_samples=100):
    """Measure diversity via average pairwise L2 distance of generated images."""
    generator.eval()
    with torch.no_grad():
        noise = torch.randn(n_samples, NZ, 1, 1, device=device)
        imgs = generator(noise).cpu().view(n_samples, -1)
    generator.train()
    
    # Compute pairwise distances (sample 500 pairs)
    n_pairs = min(500, n_samples * (n_samples - 1) // 2)
    indices = torch.randint(0, n_samples, (n_pairs, 2))
    distances = torch.norm(imgs[indices[:, 0]] - imgs[indices[:, 1]], dim=1)
    return distances.mean().item(), distances.std().item()

dc_mean, dc_std = compute_diversity_score(netG_dc)
wg_mean, wg_std = compute_diversity_score(netG_wg)

print('='*50)
print('Diversity Analysis (Avg Pairwise L2 Distance)')
print('='*50)
print(f'DCGAN:   {dc_mean:.4f} ± {dc_std:.4f}')
print(f'WGAN-GP: {wg_mean:.4f} ± {wg_std:.4f}')
print()
if wg_mean > dc_mean:
    print('✅ WGAN-GP shows HIGHER diversity — mode collapse mitigated!')
else:
    print('⚠️ DCGAN shows higher diversity — consider training WGAN-GP longer.')

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(['DCGAN', 'WGAN-GP'], [dc_mean, wg_mean],
              yerr=[dc_std, wg_std], capsize=10,
              color=['#FF5722', '#4CAF50'], edgecolor='black', linewidth=1.2)
ax.set_ylabel('Avg Pairwise L2 Distance', fontsize=12)
ax.set_title('Image Diversity Comparison', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('diversity_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 15. Save Final Models for Gradio App

In [ ]:
# Save final generator weights for the Gradio deployment
torch.save(netG_dc.state_dict(), 'dcgan_generator_final.pth')
torch.save(netG_wg.state_dict(), 'wgangp_generator_final.pth')

# Save training logs as JSON
logs = {
    'dcgan': {'G_losses': G_losses_dc, 'D_losses': D_losses_dc},
    'wgangp': {'G_losses': G_losses_wg, 'C_losses': C_losses_wg},
}
with open('training_logs.json', 'w') as f:
    json.dump(logs, f)

print('✅ Models and logs saved!')
print('  - dcgan_generator_final.pth')
print('  - wgangp_generator_final.pth')
print('  - training_logs.json')

## 16. Summary & Conclusions

In [ ]:
print('='*60)
print('           SUMMARY — Mode Collapse Analysis')
print('='*60)
print()
print(f'DCGAN  — Final G Loss: {G_losses_dc[-1]:.4f}, D Loss: {D_losses_dc[-1]:.4f}')
print(f'WGAN-GP — Final G Loss: {G_losses_wg[-1]:.4f}, C Loss: {C_losses_wg[-1]:.4f}')
print()
print(f'Diversity (L2) — DCGAN: {dc_mean:.4f}, WGAN-GP: {wg_mean:.4f}')
print()
print('Key Findings:')
print('  1. DCGAN uses BCE loss with sigmoid, which can lead to vanishing')
print('     gradients and mode collapse.')
print('  2. WGAN-GP replaces the discriminator with a critic using')
print('     Wasserstein distance, providing stable gradients throughout training.')
print('  3. Gradient Penalty (λ=10) enforces the Lipschitz constraint')
print('     without weight clipping, enabling smoother training.')
print('  4. The critic is updated 5 times per generator update,')
print('     allowing the generator to receive better gradient signals.')
print('  5. WGAN-GP generates MORE DIVERSE images, mitigating mode collapse.')